# Chapter 5: Multi-Token Prediction and FP8 Quantization

We have now established the core architectural pillars of the DeepSeek model: Multi-Head Latent Attention and Mixture-of-Experts. These innovations define *what* the model computes. Now, we turn our attention to an equally important topic that defines *how* these computations are performed with incredible efficiency.

This chapter covers two key techniques central to DeepSeek's training methodology:

1. **Multi-Token Prediction (MTP)** — A technique that provides stronger training signals by predicting multiple future tokens simultaneously
2. **FP8 Quantization** — An advanced quantization framework that trades precision for significant gains in speed and memory

This chapter is divided into two main parts. First, we will dive deep into MTP, understanding its motivation, advantages, and exactly how DeepSeek implemented their advanced, causal version of it. After mastering MTP, we will move to the second part, a deep dive into the FP8 Quantization framework.

## Required Libraries

Let's begin by importing the necessary Python libraries:

- `math`: For mathematical operations
- `typing.Optional`: For type hints with optional parameters
- `torch`: PyTorch library for tensor operations and neural networks
- `torch.nn`: Neural network modules and functions
- `torch.nn.functional`: Functional interface for neural network operations

In [ ]:
import math
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

---

# Part 1: Multi-Token Prediction (MTP)

## From Single-Token to Multi-Token Prediction

The entire training process for the language models we've discussed so far has been based on a single, simple objective: **Next-Token Prediction**. In the standard approach, the model's goal is to predict the single token that immediately follows each input token.

**Multi-Token Prediction** changes this fundamental objective. Instead of predicting only the single next token, the model is trained to predict **multiple future tokens at once**. For example, from the input token "is," it might predict the sequence "transforming," "the," "world."

### The Four Key Advantages of MTP

1. **Densification of training signals** — Each training sample contains much more information for the model to learn from, as it receives gradient signals across multiple future steps simultaneously.

2. **Improved data efficiency** — Since each training sample is more informative, the model achieves higher performance with the same amount of training data. The original MTP paper demonstrated this on coding benchmarks like MBPP and HumanEval.

3. **Better planning by prioritizing "choice points"** — The MTP loss implicitly assigns higher weights to critical tokens that significantly influence the future outcome, forcing the model to develop better internal representations for planning.

4. **Higher inference speed via speculative decoding** — MTP heads can serve as a "draft" model for speculative decoding, achieving up to 3x speedups on certain tasks.